<a href="https://colab.research.google.com/github/wlj0504/Pytorch_study/blob/master/2_2%E6%95%B0%E6%8D%AE%E9%A2%84%E5%A4%84%E7%90%86.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2.2数据预处理
核心讲解pandas库来处理原始数据转化为张量格式。

## 2.2.1读取数据集
我们先创建一个最为基础的人工数据集，并存储在CSV文件.../data/house_tiny.csv中。


In [5]:
import os

os.makedirs(os.path.join('...','data'),exist_ok=True)
data_file=os.path.join('...','data','house_tiny.csv')
with open(data_file,'w')as f:
  f.write('NumRooms,Alley,Price\n')#列名
  f.write('NA,Pave,127500\n')
  f.write('2,NA,100600\n')
  f.write('4,NA,178100\n')
  f.write('NA,NA,140000\n')


目前我们已经创建了csv文件，我们现在需要导入pandas库并调用read_csv函数，进行读取。

In [6]:
import pandas as pd
data=pd.read_csv(data_file)
print(data)

   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  100600
2       4.0   NaN  178100
3       NaN   NaN  140000


## 2.2.2 缺失值怎么处理
NaN项表示缺失值，两个方法，插值法和删除法。

插值法
iloc位置索引，data分成inputs和outputs，其中前者为data的前两列，而后者为data的最后一列。对于inputs里面缺少的数值，我们采用一列的均值来实现代替NaN。



In [10]:
inputs,outputs=data.iloc[:,0:2],data.iloc[:,2]
inputs=inputs.fillna(inputs.mean(numeric_only=True))
##numeric_only表示仅包括float，int，boolean等类型的列，不包括字符串类型的列
print(inputs)

   NumRooms Alley
0       3.0  Pave
1       2.0   NaN
2       4.0   NaN
3       3.0   NaN


注：书中的pandas是旧版的版本，而colab的版本较新，所以代码会出现一些问题，比如说：如果使用``` inputs.mean()```函数的话，新版的会对于数值的NumRooms进行求和平均来填补NaN，同时也会对于Alley这一列的字符串求和平均，这时候就会报错。

In [21]:
inputs=pd.get_dummies(inputs,dummy_na=True,dtype=int)
print(inputs)
print(inputs.dtypes)

   NumRooms  Alley_Pave  Alley_nan
0       3.0        True      False
1       2.0       False       True
2       4.0       False       True
3       3.0       False       True
NumRooms      float64
Alley_Pave       bool
Alley_nan        bool
dtype: object


注：新版旧版的pandas库不一样，新版会生成true或者false的布尔类型变量，但是旧版的默认生成整数类型(0/1),emmm当前的这个生成结果可能是bug


## 2.2.3 转换成为这样的张量格式
目前的inputs和outputs都是数值类型，我们可以转换成为张量类型。

In [27]:
import torch
X=torch.tensor(inputs.to_numpy(dtype=float))
Y=torch.tensor(outputs.to_numpy(dtype=float))
print(X)
print(Y)
Y=torch.tensor(outputs.to_numpy(dtype=float)).reshape(-1,1)#这里的-1并不是表示负数的意思，而是告诉Python这一维度需要自动推算
Y#colab默认只显示最后一行结果

tensor([[3., 1., 0.],
        [2., 0., 1.],
        [4., 0., 1.],
        [3., 0., 1.]], dtype=torch.float64)
tensor([127500., 100600., 178100., 140000.], dtype=torch.float64)


tensor([[127500.],
        [100600.],
        [178100.],
        [140000.]], dtype=torch.float64)